In [ ]:
import sys; sys.path.insert(0, '../src')
import numpy as np
from models.graphene import SingleLayerGrapheneTB, SingleLayerGrapheneKP
from response.dos import (compute_dos, compute_dos_vectorized,
                           dirac_dos_analytical, generate_dirac_mesh,
                           integrate_bz)
vF = np.sqrt(3)/2 * 2.78  # ~2.4076
# ── 1. TB 全BZ ────────────────────────────────────
print('=== 1. TB full-BZ DOS ===')
tb = SingleLayerGrapheneTB(t=2.78)
print(f'eta: {0.03}, Δe: {8.5/1200}')

In [ ]:
E_vals, dos = compute_dos(tb, nk=150, sigma=0.03,
                           E_range=(-8.5, 8.5), nE=2400)
de = E_vals[1] - E_vals[0]
total = np.sum(dos) * de
print(f'  Integral: {total:.3f} (expect 4)')
# Van Hove
vhs_idx = np.argmax(dos)
print(f'  VHS peak at E={E_vals[vhs_idx]:.2f} eV, DOS={dos[vhs_idx]:.3f}')
# Dirac 线性
mask = (E_vals > 0.05) & (E_vals < 0.8)
slope = np.polyfit(E_vals[mask], dos[mask], 1)[0]
sl_an = dirac_dos_analytical(1.0, vF)
print(f'  Dirac slope: {slope:.3f} (analytical: {sl_an:.3f}), ratio={slope/sl_an:.3f}')


In [ ]:
# ── 2. 矢量化 vs 循环一致 ─────────────────────────
E2, d2 = compute_dos_vectorized(tb, nk=150, sigma=0.03,
                                 E_range=(-8.5, 8.5), nE=2400)
diff = np.max(np.abs(dos - d2))
print(f'  Vec vs Loop max diff: {diff:.2e}')


In [ ]:

import matplotlib.pyplot as plt

plt.style.use('bmh')

%matplotlib inline
%config InlineBackend.figure_format='retina'
import matplotlib as mpl
mpl.rcParams.update({
    'font.size': 10.5,
    'font.family': ['Times New Roman','SimSun'],# 'HP Simplified Hans', # Tahoma
    'axes.unicode_minus': False,
    'axes.facecolor': 'white',
    'figure.facecolor': 'white',
    'savefig.facecolor': 'white', # optional: saved figures
    # 'grid.color': '#e0e0e0', # optional: lighter grid
    'axes.edgecolor': 'black', # optional: clearer axes
})

fig, ax = plt.subplots(figsize=(4,3))
ax.plot(E_vals, dos) # 3.3e7 for i=31; 8e6 for i=15
ax.set_xbound(E2[0], E2[-1])
ax.set_xlabel(r"Energy (eV)") # $\gamma_0$
ax.set_ylabel("DOS") #  (states/eV·unit cell)
ax.set_title(''.join([r'$\eta$=', f'{0.03:.2g} ',r'eV; $\Delta E$=',f'{8.5/1200:.3g} ','eV']), loc='right') #$\gamma_0$
fig.tight_layout()
plt.show()

In [ ]:
# ── 3. KP Dirac 中心网格 ──────────────────────────
# print()
print('=== 2. KP Dirac-centered mesh ===')
kp = SingleLayerGrapheneKP(t=2.78, valley=+1)
k_mesh = generate_dirac_mesh(kp, nk=100, k_cut=0.5, valley='K')
area = (2*0.5)**2
# KP 单谷模型需手动设 g=4 (自旋+谷)
kp_orig_g = kp.degeneracy_factor()
kp.valley_degeneracy = 2
E_kp, d_kp = compute_dos(kp, k_cart=k_mesh, sigma=0.005,
                          E_range=(-1.2, 1.2), nE=900, k_area=area)
kp.valley_degeneracy = 1  # 恢复
sl_kp = np.polyfit(E_kp[E_kp>0.05][:30], d_kp[E_kp>0.05][:30], 1)[0]
print(f'  KP Dirac slope: {sl_kp:.3f} (analytical: {sl_an:.3f}), ratio={sl_kp/sl_an:.3f}')
mid = len(d_kp)//2
print(f'  DOS(0)={d_kp[mid]:.6f}, DOS(0.5)={d_kp[mid+62]:.4f}')


In [ ]:
# ── 4. 通用 BZ 积分: 验证 Σ_n ∫ dE DOS(E) = N_states
# print()
print('=== 3. integrate_bz: total states ===')
def count_states(k, E_k_local, V_k_local):
    return float(len(E_k_local))  # n_bands per k-point
n_states = integrate_bz(tb, count_states, nk=80)
print(f'  Total states via integrate_bz: {n_states:.3f} (expect 4)')
print()
print('All checks done.')